# Diffusion + Structure Conditioning — 128×128  (Model D)
### Model D: an iterative **denoising diffusion** model, structure-conditioned, with **classifier-free guidance**

Models A/B/C are GANs (single-shot generators). **Model D is a different idea:** it
learns to *iteratively denoise*. It **keeps Model C's winning insight** — condition
on a per-image **structure map** (Canny + silhouette + distance transform; aligned
target, 100% coverage) — but replaces the adversarial generator with a
**time-conditioned U-Net** trained with a simple noise-prediction MSE (DDPM).

**Why this should be *more accurate*:**
- **Diffusion > GAN** on class-conditional fidelity, and it trains **stably** (no
  G/D balancing, no mode collapse) — ideal for small/reduced runs.
- **ControlNet-lite conditioning:** the structure map is concatenated to the noisy
  image at the U-Net input, so D inherits C's exact-shape control.
- **Classifier-free guidance (CFG)** is the accuracy dial: the label is dropped
  ~10% of training steps (so the model learns conditional **and** unconditional
  scores); at sampling we push toward the class with a guidance scale `w`. Higher
  `w` ⇒ stronger class adherence ⇒ higher **GAN-test recognition**. Section 10
  sweeps `w` to show the trade-off.
- **DDIM** sampling (few steps) keeps generation tractable.

This is the literature's consensus next step beyond a structure-conditioned cGAN
(pix2pix → ControlNet-style diffusion; SignDiff / Sign-IDD for sign language) and
the direction `reports/conclusions.md` recommends. **Dependency:** OpenCV only
(no MediaPipe).

## Section 1 — Environment

In [ ]:
# ── Environment detection (Colab vs local Linux/WSL) ────────────────────────────
import os, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def _find_repo_root(start):
    d = os.path.abspath(start)
    for _ in range(5):
        if os.path.isdir(os.path.join(d, 'data')) and os.path.isdir(os.path.join(d, 'notebooks')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return os.path.abspath(start)

REPO_ROOT = None if IN_COLAB else _find_repo_root(os.getcwd())
print('Environment:', 'Google Colab' if IN_COLAB else f'Local ({sys.platform}) - repo root: {REPO_ROOT}')


In [ ]:
# ── Install (pinned for reproducibility) ──────────────────────────────────────
# Model D needs NO MediaPipe — structure maps come from OpenCV; diffusion is pure TF.
if IN_COLAB:
    !pip install tensorflow==2.19.0 opencv-python --quiet
    !pip install scikit-image scikit-learn tqdm --quiet
    print("All packages installed.")
else:
    print('Running locally -- make sure this environment has: tensorflow==2.19.0 (tensorflow[and-cuda] for GPU on Linux/WSL), opencv-python, scikit-image, scikit-learn, tqdm.')


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Not running in Colab -- skipping Drive mount; using local paths.')


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import os, json, warnings, math
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from skimage.metrics import structural_similarity as ssim_fn
import cv2

warnings.filterwarnings('ignore', category=UserWarning)
print(f"TensorFlow : {tf.__version__}")
print(f"OpenCV     : {cv2.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── Global reproducibility ─────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED); tf.random.set_seed(RANDOM_SEED)
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError: pass
print(f"Global seed: {RANDOM_SEED} | GPUs: {gpus}")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DRIVE_BASE     = "/content/drive/MyDrive/diffusion_D_128struct" if IN_COLAB else os.path.join(REPO_ROOT, "runs", "diffusion_D_128struct")
DATA_PATH      = "/content/ArASL_Database_54K_Final" if IN_COLAB else os.path.join(REPO_ROOT, "data", "ArASL_dataset")
CHECKPOINT_DIR = os.path.join(DRIVE_BASE, "checkpoints")
SAMPLES_DIR    = os.path.join(DRIVE_BASE, "samples")
PLOTS_DIR      = os.path.join(DRIVE_BASE, "plots")
HISTORY_DIR    = os.path.join(DRIVE_BASE, "history")
for d in [CHECKPOINT_DIR, SAMPLES_DIR, PLOTS_DIR, HISTORY_DIR]:
    os.makedirs(d, exist_ok=True)
print("Directories ready under", DRIVE_BASE)


In [ ]:
# ── Dataset — auto-download from Hugging Face (no manual Drive upload needed) ──
# Source: pain/ArASL_Database_Grayscale on Hugging Face (54,049 imgs, 32 classes).
# Downloads straight to local Colab disk once per session — faster than reading
# from Drive and doesn't touch your Drive quota. Re-running is a no-op if present.
import pandas as pd

if not os.path.isdir(DATA_PATH) or not os.listdir(DATA_PATH):
    import io as _io, json as _json, urllib.request as _req
    try:
        import pyarrow.parquet as pq
    except ImportError:
        import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow"]); import pyarrow.parquet as pq

    _parquet = "/content/arasl.parquet"
    if not os.path.exists(_parquet):
        print("Downloading ArASL dataset from Hugging Face (~30 MB)...")
        _req.urlretrieve(
            "https://huggingface.co/api/datasets/pain/ArASL_Database_Grayscale/parquet/default/train/0.parquet",
            _parquet)

    _names = ["ain","al","aleff","bb","dal","dha","dhad","fa","gaaf","ghain","ha","haa",
              "jeem","kaaf","khaa","la","laam","meem","nun","ra","saad","seen","sheen",
              "ta","taa","thaa","thal","toot","waw","ya","yaa","zay"]
    try:
        _md = pq.read_schema(_parquet).metadata or {}
        _hf = _md.get(b"huggingface")
        if _hf:
            _names = _json.loads(_hf)["info"]["features"]["label"]["names"]
    except Exception as e:
        print("metadata note:", e)

    os.makedirs(DATA_PATH, exist_ok=True)
    for n in _names:
        os.makedirs(os.path.join(DATA_PATH, n), exist_ok=True)

    _df = pd.read_parquet(_parquet)
    _counts = {}
    for v, lab in zip(_df["image"], _df["label"]):
        b = v["bytes"] if isinstance(v, dict) else v
        cls = _names[int(lab)] if int(lab) < len(_names) else f"class_{int(lab):02d}"
        k = _counts.get(cls, 0); _counts[cls] = k + 1
        Image.open(_io.BytesIO(b)).convert("L").save(os.path.join(DATA_PATH, cls, f"{cls}_{k:04d}.png"))
    print(f"Dataset ready: {sum(_counts.values())} images across {len(_counts)} classes -> {DATA_PATH}")
else:
    print(f"Dataset already present at {DATA_PATH} — skipping download.")


## Section 2 — Hyperparameters

In [ ]:
# ════════════════════════════════════════════════════════
#  CONFIGURATION — change only this cell
# ════════════════════════════════════════════════════════
IMG_SIZE     = 128
IMG_CHANNELS = 1
COND_CH      = 3        # structure map: edge + silhouette + distance

# Diffusion
T_TRAIN      = 1000     # training noise steps
DDIM_STEPS   = 50       # sampling steps (raise for quality, lower for speed)
P_UNCOND     = 0.1      # label-dropout prob -> enables classifier-free guidance
CFG_EVAL     = 3.0      # guidance scale for headline samples / recognition
CFG_SWEEP    = [0.0, 1.0, 2.0, 3.0, 5.0]

# Training
EPOCHS       = 60
BATCH_SIZE   = 32
LR           = 2e-4
EMB_DIM      = 256
BASE_CH      = 48

# Held-out split (structure generalization test)
EVAL_FRACTION, MIN_EVAL_PER_CLS = 0.10, 30
# Canny thresholds
CANNY_LO, CANNY_HI = 60, 160
SAVE_EVERY_N = 5

print("Model D — structure-conditioned diffusion + classifier-free guidance")
print(f"  IMG_SIZE={IMG_SIZE} COND_CH={COND_CH} | T_TRAIN={T_TRAIN} DDIM_STEPS={DDIM_STEPS}")
print(f"  P_UNCOND={P_UNCOND} (CFG enabled) | CFG_EVAL w={CFG_EVAL} | sweep {CFG_SWEEP}")

## Section 3 — Data Loading

In [ ]:
# ── Load images at 128×128 ─────────────────────────────────────────────────────
VALID_EXT = ('.png', '.jpg', '.jpeg')
images_list, labels_list = [], []
print(f"Loading from: {DATA_PATH}  (resizing to {IMG_SIZE}×{IMG_SIZE})")
for subfolder in tqdm(sorted(os.listdir(DATA_PATH)), desc="Classes"):
    spath = os.path.join(DATA_PATH, subfolder)
    if not os.path.isdir(spath): continue
    for fname in sorted(os.listdir(spath)):
        if not fname.lower().endswith(VALID_EXT): continue
        try:
            raw = tf.io.read_file(os.path.join(spath, fname))
            img = tf.image.decode_png(raw, channels=1)
            img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
            img = (tf.cast(img, tf.float32) - 127.5) / 127.5      # [-1, 1]
            images_list.append(img.numpy()); labels_list.append(subfolder)
        except Exception:
            pass
print(f"Loaded : {len(images_list)} images")

In [ ]:
# ── Arrays + label encoding + held-out split ──────────────────────────────────
images_array = np.array(images_list, dtype=np.float32)
label_encoder = LabelEncoder()
labels_int = label_encoder.fit_transform(np.array(labels_list)).astype(np.int64)
num_classes = len(label_encoder.classes_)
NULL_CLASS  = num_classes                       # "unconditional" embedding row for CFG
idx_to_label = {i: l for i, l in enumerate(label_encoder.classes_)}
label_to_idx = {l: i for i, l in enumerate(label_encoder.classes_)}
np.save(os.path.join(CHECKPOINT_DIR, "classes.npy"), label_encoder.classes_)

rng = np.random.default_rng(RANDOM_SEED); perm = rng.permutation(len(images_array))
images_array, labels_int = images_array[perm], labels_int[perm]
n_eval = max(num_classes * MIN_EVAL_PER_CLS, int(EVAL_FRACTION * len(images_array)))
X_tr, y_tr = images_array[:-n_eval], labels_int[:-n_eval]
X_ev, y_ev = images_array[-n_eval:], labels_int[-n_eval:]
print(f"classes {num_classes} | train {X_tr.shape} | held-out {X_ev.shape}")

## Section 4 — Structure Maps (the conditioning signal, shared with Model C)

Canny edges + Otsu silhouette + distance transform → a 3-channel map with **100%
coverage**. The U-Net is conditioned on this map.

In [ ]:
def structure_map(img_norm):
    """(H,W,1) in [-1,1] -> (H,W,3) structure map in [-1,1]."""
    g = ((img_norm[:, :, 0] + 1) * 127.5).clip(0, 255).astype(np.uint8)
    edge = cv2.Canny(g, CANNY_LO, CANNY_HI)
    _, sil = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if sil.mean() > 127: sil = 255 - sil
    dist = cv2.normalize(cv2.distanceTransform(sil, cv2.DIST_L2, 3), None, 0, 255, cv2.NORM_MINMAX)
    return (np.stack([edge, sil, dist], -1).astype(np.float32) / 127.5) - 1.0

C_tr = np.stack([structure_map(x) for x in tqdm(X_tr, desc="structure maps")]).astype(np.float32)
print("condition maps:", C_tr.shape, "| coverage", float(np.mean([np.any(c > -0.99) for c in C_tr])))

## Section 5 — Diffusion Schedule + Time/Class-Conditioned U-Net

A cosine β-schedule and a U-Net that predicts the noise `ε`. The **structure map is
concatenated to the noisy image** at the input (ControlNet-lite); the **timestep**
and **class** enter through a shared embedding added at every block.

In [ ]:
# ── Cosine schedule ───────────────────────────────────────────────────────────
def cosine_alpha_bar(T, s=0.008):
    t = np.linspace(0, T, T + 1) / T
    f = np.cos((t + s) / (1 + s) * np.pi / 2) ** 2
    ab = f / f[0]
    betas = np.clip(1 - ab[1:] / ab[:-1], 1e-4, 0.999).astype(np.float32)
    return betas
betas = cosine_alpha_bar(T_TRAIN)
alpha_bar = np.cumprod(1.0 - betas).astype(np.float32)
ab_tf = tf.constant(alpha_bar)

def sinusoidal_emb(t, dim=EMB_DIM):
    half = dim // 2
    freqs = tf.exp(-math.log(10000.0) * tf.range(half, dtype=tf.float32) / (half - 1))
    args = tf.cast(t[:, None], tf.float32) * freqs[None, :]
    return tf.concat([tf.sin(args), tf.cos(args)], -1)

def q_sample(x0, t, noise):
    ab = tf.gather(ab_tf, t)[:, None, None, None]
    return tf.sqrt(ab) * x0 + tf.sqrt(1.0 - ab) * noise

In [ ]:
# ── Time/class-conditioned U-Net (structure map concatenated at input) ─────────
def cblock(x, f):
    return layers.Activation("swish")(layers.GroupNormalization(8)(layers.Conv2D(f, 3, padding="same")(x)))

def build_unet():
    img  = tf.keras.Input((IMG_SIZE, IMG_SIZE, IMG_CHANNELS))   # noisy x_t
    cond = tf.keras.Input((IMG_SIZE, IMG_SIZE, COND_CH))        # structure map
    t    = tf.keras.Input((), dtype=tf.int32)
    y    = tf.keras.Input((), dtype=tf.int32)                   # class id (num_classes = null)
    temb = layers.Dense(EMB_DIM, activation="swish")(sinusoidal_emb(tf.cast(t, tf.float32)))
    yemb = layers.Embedding(num_classes + 1, EMB_DIM)(y)
    emb  = layers.Dense(EMB_DIM, activation="swish")(temb + yemb)
    def add_emb(h, f): return h + layers.Dense(f)(emb)[:, None, None, :]

    x  = cblock(layers.Concatenate()([img, cond]), BASE_CH)              # 128
    d1 = add_emb(cblock(x,  BASE_CH),   BASE_CH);    p1 = layers.AveragePooling2D()(d1)   # 64
    d2 = add_emb(cblock(p1, BASE_CH*2), BASE_CH*2);  p2 = layers.AveragePooling2D()(d2)   # 32
    d3 = add_emb(cblock(p2, BASE_CH*4), BASE_CH*4);  p3 = layers.AveragePooling2D()(d3)   # 16
    d4 = add_emb(cblock(p3, BASE_CH*8), BASE_CH*8);  p4 = layers.AveragePooling2D()(d4)   # 8
    b  = add_emb(cblock(p4, BASE_CH*8), BASE_CH*8);  b  = cblock(b, BASE_CH*8)
    u4 = layers.UpSampling2D()(b);  u4 = add_emb(cblock(layers.Concatenate()([u4, d4]), BASE_CH*8), BASE_CH*8)
    u3 = layers.UpSampling2D()(u4); u3 = add_emb(cblock(layers.Concatenate()([u3, d3]), BASE_CH*4), BASE_CH*4)
    u2 = layers.UpSampling2D()(u3); u2 = add_emb(cblock(layers.Concatenate()([u2, d2]), BASE_CH*2), BASE_CH*2)
    u1 = layers.UpSampling2D()(u2); u1 = add_emb(cblock(layers.Concatenate()([u1, d1]), BASE_CH), BASE_CH)
    out = layers.Conv2D(IMG_CHANNELS, 3, padding="same", dtype="float32")(u1)   # predicts eps
    return tf.keras.Model([img, cond, t, y], out, name="unet_D")

unet = build_unet()
print(f"U-Net params: {unet.count_params():,}")

## Section 6 — Training (DDPM noise-prediction + classifier-free dropout)

In [ ]:
opt = tf.keras.optimizers.Adam(LR)

@tf.function
def train_step(x0, cond, y):
    bs = tf.shape(x0)[0]
    t  = tf.random.uniform([bs], 0, T_TRAIN, dtype=tf.int32)
    noise = tf.random.normal(tf.shape(x0))
    xt = q_sample(x0, t, noise)
    drop = tf.cast(tf.random.uniform([bs]) < P_UNCOND, tf.int32)   # CFG: drop label
    y_in = y * (1 - drop) + NULL_CLASS * drop
    with tf.GradientTape() as tape:
        pred = unet([xt, cond, t, y_in], training=True)
        loss = tf.reduce_mean(tf.square(pred - noise))
    opt.apply_gradients(zip(tape.gradient(loss, unet.trainable_variables), unet.trainable_variables))
    return loss

def make_ds():
    ds = tf.data.Dataset.from_tensor_slices((X_tr, C_tr, y_tr.astype(np.int32)))
    return ds.shuffle(len(X_tr), seed=RANDOM_SEED, reshuffle_each_iteration=True)\
             .batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
train_ds = make_ds()

In [ ]:
# ── Run training ──────────────────────────────────────────────────────────────
hist = {"loss": []}
print(f"Training Model D — {EPOCHS} epochs")
for ep in range(1, EPOCHS + 1):
    tot = k = 0.0
    for x0, cond, y in train_ds:
        tot += float(train_step(x0, cond, y)); k += 1
    hist["loss"].append(tot / k)
    print(f"  ep{ep:02d}/{EPOCHS}  loss={tot/k:.4f}")
    if ep % SAVE_EVERY_N == 0 or ep == EPOCHS:
        unet.save_weights(os.path.join(CHECKPOINT_DIR, "unet_D.weights.h5"))
print("Training complete.")

## Section 7 — DDIM Sampler with Classifier-Free Guidance

In [ ]:
def ddim_sample(net, cond, y, w=CFG_EVAL, steps=DDIM_STEPS, seed=0):
    """cond: (B,H,W,3) structure maps; y: (B,) class ids. Returns (B,H,W,1) in [-1,1]."""
    bs = cond.shape[0]
    tf.random.set_seed(seed)
    x = tf.random.normal([bs, IMG_SIZE, IMG_SIZE, IMG_CHANNELS])
    y_cond = tf.constant(y, tf.int32); y_null = tf.fill([bs], NULL_CLASS)
    ts = np.linspace(T_TRAIN - 1, 0, steps).astype(np.int32)
    for i, tcur in enumerate(ts):
        tb = tf.fill([bs], int(tcur))
        eps_c = net([x, cond, tb, y_cond], training=False)
        if w != 0.0:
            eps_u = net([x, cond, tb, y_null], training=False)
            eps = eps_u + w * (eps_c - eps_u)          # classifier-free guidance
        else:
            eps = eps_c
        ab_t = alpha_bar[tcur]
        x0 = tf.clip_by_value((x - math.sqrt(1 - ab_t) * eps) / math.sqrt(ab_t), -1.0, 1.0)
        if i < len(ts) - 1:
            ab_n = alpha_bar[ts[i + 1]]
            x = math.sqrt(ab_n) * x0 + math.sqrt(1 - ab_n) * eps
        else:
            x = x0
    return x.numpy()

In [ ]:
# ── Visual: condition on held-out structures, sample at w=CFG_EVAL ────────────
TEST_LABELS = [l for l in ["bb", "ain", "gaaf", "al", "ha"] if l in label_to_idx] or \
              [idx_to_label[i] for i in range(min(5, num_classes))]
_vis = [int((np.where(y_ev == label_to_idx[l])[0])[0]) for l in TEST_LABELS]
VIS_COND = np.stack([structure_map(X_ev[i]) for i in _vis]).astype(np.float32)
samp = ddim_sample(unet, tf.constant(VIS_COND), np.array([label_to_idx[l] for l in TEST_LABELS]), w=CFG_EVAL)
fig, ax = plt.subplots(1, len(TEST_LABELS), figsize=(4*len(TEST_LABELS), 4))
for i, l in enumerate(TEST_LABELS):
    ax[i].imshow((samp[i,:,:,0]*127.5+127.5).clip(0,255).astype('uint8'), cmap='gray')
    ax[i].set_title(l, fontweight='bold'); ax[i].axis('off')
plt.suptitle(f"Model D — DDIM samples (CFG w={CFG_EVAL})", y=1.02); plt.tight_layout(); plt.show()

## Section 8 — Loss Curve

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(range(1, len(hist["loss"])+1), hist["loss"])
plt.title("Model D — denoising MSE"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.grid(alpha=.3)
plt.savefig(os.path.join(PLOTS_DIR, "loss_D.png"), dpi=150, bbox_inches="tight"); plt.show()

## Section 9 — Evaluation: GAN-test, Diversity, and the **CFG Sweep**

Same recognition/diversity protocol as A/B/C (classifier trained on real, tested on
generated). The headline new result is the **CFG sweep**: recognition vs guidance
scale `w` — the knob that buys accuracy.

In [ ]:
# ── Reference classifier on REAL (for GAN-test recognition) ───────────────────
clf = tf.keras.Sequential([tf.keras.Input((IMG_SIZE, IMG_SIZE, 1)),
    layers.Conv2D(32,3,2,"same"), layers.LeakyReLU(0.2),
    layers.Conv2D(64,3,2,"same"), layers.LeakyReLU(0.2),
    layers.Conv2D(128,3,2,"same"), layers.LeakyReLU(0.2),
    layers.GlobalAveragePooling2D(), layers.Dense(num_classes, dtype="float32")])
clf.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
clf.fit(X_tr, y_tr, epochs=8, batch_size=64, validation_data=(X_ev, y_ev), verbose=0)
clf_ref = float(clf.evaluate(X_ev, y_ev, verbose=0)[1])
print(f"classifier real held-out acc: {clf_ref:.4f}  (chance {1/num_classes:.3f})")

In [ ]:
def gen_from_train(net, w, n_per=40, seed=0):
    out, ys = [], []
    for c in range(num_classes):
        pool = np.where(y_tr == c)[0]
        ci = np.random.default_rng(seed + c).choice(pool, n_per)
        out.append(ddim_sample(net, tf.constant(C_tr[ci]), np.full(n_per, c), w=w, seed=seed*10+c))
        ys.append(np.full(n_per, c))
    return np.concatenate(out), np.concatenate(ys)

def recognition(net, w, n_per=40):
    f, ys = gen_from_train(net, w, n_per)
    return float((clf.predict(f, verbose=0).argmax(1) == ys).mean())

def diversity(net, w, n_per=12, seed=1):
    ds = []
    for c in range(num_classes):
        pool = np.where(y_tr == c)[0]; ci = np.random.default_rng(seed+c).choice(pool, n_per)
        f = ddim_sample(net, tf.constant(C_tr[ci]), np.full(n_per, c), w=w, seed=seed+c).reshape(n_per, -1)
        ds += [float(np.mean(np.abs(f[i]-f[j]))) for i in range(n_per) for j in range(i+1, n_per)]
    return float(np.mean(ds))

# ── CFG sweep ─────────────────────────────────────────────────────────────────
sweep = {}
for w in CFG_SWEEP:
    r = recognition(unet, w, n_per=24)
    sweep[w] = r
    print(f"  CFG w={w:>3}: recognition={r:.4f}")
best_w = max(sweep, key=sweep.get)
print(f"best recognition {sweep[best_w]:.4f} at w={best_w}")

In [ ]:
# ── Headline metrics at CFG_EVAL + plot the sweep ─────────────────────────────
rec_m = recognition(unet, CFG_EVAL, n_per=40)
div_m = diversity(unet, CFG_EVAL)
print(f"GAN-test recognition @ w={CFG_EVAL}: {rec_m:.4f}")
print(f"Diversity            @ w={CFG_EVAL}: {div_m:.4f}")
plt.figure(figsize=(6,4))
ws = list(sweep.keys()); plt.plot(ws, [sweep[w] for w in ws], "o-")
plt.axhline(clf_ref, ls="--", c="gray", label=f"real ref {clf_ref:.2f}")
plt.title("Model D — recognition vs CFG scale"); plt.xlabel("guidance w"); plt.ylabel("GAN-test recognition")
plt.legend(); plt.grid(alpha=.3); plt.savefig(os.path.join(PLOTS_DIR, "cfg_sweep_D.png"), dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# ── ⭐ Held-out structure test (does D generalize like C?) ────────────────────
C_ev = np.stack([structure_map(x) for x in X_ev]).astype(np.float32)
fake_ev = ddim_sample(unet, tf.constant(C_ev), y_ev, w=CFG_EVAL, seed=0)
rec_heldout = float((clf.predict(fake_ev, verbose=0).argmax(1) == y_ev).mean())
ssims = [float(ssim_fn((X_ev[i,:,:,0]+1)/2, (fake_ev[i,:,:,0]+1)/2, data_range=1.0)) for i in range(len(X_ev))]
print(f"Held-out recognition: {rec_heldout:.4f}  | fidelity SSIM: {float(np.mean(ssims)):.4f}")
# qualitative grid
n_show = min(6, len(X_ev))
fig, axes = plt.subplots(3, n_show, figsize=(2.4*n_show, 7.2))
for j in range(n_show):
    axes[0,j].imshow((C_ev[j]*127.5+127.5).clip(0,255).astype('uint8')); axes[0,j].set_title(idx_to_label[y_ev[j]]); axes[0,j].axis('off')
    axes[1,j].imshow((fake_ev[j,:,:,0]*127.5+127.5).clip(0,255).astype('uint8'), cmap='gray'); axes[1,j].axis('off')
    axes[2,j].imshow((X_ev[j,:,:,0]*127.5+127.5).clip(0,255).astype('uint8'), cmap='gray'); axes[2,j].axis('off')
for r, lab in enumerate(["unseen structure", "D output", "real target"]):
    axes[r,0].axis('on'); axes[r,0].set_xticks([]); axes[r,0].set_yticks([]); axes[r,0].set_ylabel(lab)
plt.suptitle("Model D — held-out structure test", y=1.01); plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "heldout_D.png"), dpi=150, bbox_inches="tight"); plt.show()

## Section 10 — Save Results

In [ ]:
def _nn(o):
    if isinstance(o, float) and o != o: return None
    if isinstance(o, dict):  return {k: _nn(v) for k, v in o.items()}
    if isinstance(o, list):  return [_nn(i) for i in o]
    return o

results_D = {
    "model": "Diffusion-D 128×128 (structure-conditioned + classifier-free guidance)",
    "approach": "DDPM noise-prediction U-Net, structure map concatenated at input, CFG label-dropout",
    "recognition": {"gan_test_at_cfg": rec_m, "cfg_scale": CFG_EVAL,
                    "classifier_ref_on_real": clf_ref,
                    "cfg_sweep": {str(k): v for k, v in sweep.items()},
                    "best_recognition": sweep[best_w], "best_cfg": best_w},
    "diversity": {"mean": div_m},
    "heldout_structure_test": {"heldout_recognition": rec_heldout, "fidelity_ssim": float(np.mean(ssims))},
    "training": {"final_loss": hist["loss"][-1] if hist["loss"] else None},
    "hyperparams": {"IMG_SIZE": IMG_SIZE, "T_TRAIN": T_TRAIN, "DDIM_STEPS": DDIM_STEPS,
                    "P_UNCOND": P_UNCOND, "EPOCHS": EPOCHS, "CFG_EVAL": CFG_EVAL},
}
rp = os.path.join(HISTORY_DIR, "D_results.json")
with open(rp, "w") as f: json.dump(_nn(results_D), f, indent=2)
print("Saved", rp, "\n")
print(f"GAN-test recognition @ w={CFG_EVAL}: {rec_m:.4f}  (real ref {clf_ref:.4f})")
print(f"CFG sweep            : " + "  ".join(f"w{k}={v:.3f}" for k, v in sweep.items()))
print(f"Held-out recognition : {rec_heldout:.4f}  (SSIM {float(np.mean(ssims)):.4f})")
print()
print("Model D is an iterative diffusion model (different from the A/B/C GANs). It keeps")
print("Model C's structure conditioning but adds classifier-free guidance as a tunable")
print("accuracy dial -- higher CFG w trades diversity for stronger class recognition.")